In [19]:
# Imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
from PIL import Image
import os
import numpy as np
import pickle
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


In [20]:
# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Transforms
ssl_transform = transforms.Compose([
    transforms.RandomResizedCrop(size=1024),
    transforms.RandomHorizontalFlip(),
    transforms.RandomApply([transforms.ColorJitter(0.4,0.4,0.4,0.1)], p=0.8),
    transforms.RandomGrayscale(p=0.2),
    transforms.GaussianBlur(3),
    transforms.ToTensor()
])

eval_transform = transforms.Compose([
    transforms.Resize((1024,1024)),
    transforms.ToTensor()
])


Using device: cpu


In [21]:
# SSL Dataset
class SSLDataset(torch.utils.data.Dataset):
    def __init__(self, root, transform):
        self.dataset = datasets.ImageFolder(root=root, transform=None)
        self.transform = transform

    def __getitem__(self, idx):
        path, label = self.dataset.samples[idx]
        image = self.dataset.loader(path)
        x1 = self.transform(image)
        x2 = self.transform(image)
        return x1, x2, label

    def __len__(self):
        return len(self.dataset)


In [22]:
# Load Data
#train_data = SSLDataset(root="data/train", transform=ssl_transform)
#train_loader = DataLoader(train_data, batch_size=4, shuffle=True, drop_last=True)

#eval_data = datasets.ImageFolder(root="data/train", transform=eval_transform)
#eval_loader = DataLoader(eval_data, batch_size=4, shuffle=False)
#class_names = eval_data.classes
#print("Classes:", class_names)

train_data = SSLDataset(root="data/train", transform=ssl_transform)
train_loader = DataLoader(train_data, batch_size=2, shuffle=True, drop_last=True, pin_memory=True)

eval_data = datasets.ImageFolder(root="data/train", transform=eval_transform)
eval_loader = DataLoader(eval_data, batch_size=2, shuffle=False, pin_memory=True)

class_names = eval_data.classes
print("Classes:", class_names)

Classes: ['fake', 'real']


In [23]:
# SimCLR Model
class SimCLR(nn.Module):
    def __init__(self, base_encoder, projection_dim=128):
        super(SimCLR, self).__init__()
        self.encoder = nn.Sequential(*list(base_encoder.children())[:-1])  # remove the FC layer
        in_features = base_encoder.fc.in_features

        # Projection head (2-layer MLP)
        self.projector = nn.Sequential(
            nn.Linear(in_features, in_features),
            nn.ReLU(),
            nn.Linear(in_features, projection_dim)
        )

    def forward(self, x):
        h = self.encoder(x)
        h = torch.flatten(h, start_dim=1)
        z = self.projector(h)
        return h, z

In [24]:
# NT-Xent Loss
def nt_xent_loss(z1, z2, temperature=0.5):
    batch_size = z1.shape[0]
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)
    representations = torch.cat([z1, z2], dim=0)
    sim_matrix = torch.matmul(representations, representations.T)
    mask = torch.eye(2 * batch_size, dtype=torch.bool, device=z1.device)
    sim_matrix = sim_matrix[~mask].view(2 * batch_size, -1)
    positives = torch.cat([torch.sum(z1 * z2, dim=-1),
                           torch.sum(z2 * z1, dim=-1)], dim=0).unsqueeze(1)
    logits = torch.cat([positives, sim_matrix], dim=1) / temperature
    labels = torch.zeros(2 * batch_size, dtype=torch.long, device=z1.device)
    return F.cross_entropy(logits, labels)


In [25]:
# SSL Training
def train_ssl(model, loader, epochs=10, lr=3e-4, device="cpu"):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for x1, x2, _ in loader:
            x1, x2 = x1.to(device), x2.to(device)
            _, z1 = model(x1)
            _, z2 = model(x2)
            loss = nt_xent_loss(z1, z2)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {total_loss/len(loader):.4f}")

    torch.save(model.state_dict(), "ssl_encoder.pth")
    print("✅ Encoder saved as ssl_encoder.pth")
    return model

In [26]:
# Save Embeddings for Downstream Evaluation
def save_ssl_features(model, loader, device="cpu", output_path="ssl_features.pkl"):
    model = model.to(device)
    model.eval()
    all_features, all_labels = [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            h, _ = model(images)
            all_features.append(h.cpu().numpy())
            all_labels.append(labels.numpy())

    all_features = np.concatenate(all_features)
    all_labels = np.concatenate(all_labels)
    data = {"features": all_features, "labels": all_labels}

    with open(output_path, "wb") as f:
        pickle.dump(data, f)
    print(f"✅ Saved SSL features to {output_path}")


In [27]:
# Initialize and Train SSL
base_encoder = models.resnet18(weights=None)
simclr = SimCLR(base_encoder)
simclr = train_ssl(simclr, train_loader, epochs=30, device=device)

Epoch [1/30], Loss: 1.1907
Epoch [2/30], Loss: 1.1584
Epoch [3/30], Loss: 1.1428
Epoch [4/30], Loss: 1.1675
Epoch [5/30], Loss: 1.1231
Epoch [6/30], Loss: 1.1485
Epoch [7/30], Loss: 1.1512
Epoch [8/30], Loss: 1.1450
Epoch [9/30], Loss: 1.1590
Epoch [10/30], Loss: 1.1088
Epoch [11/30], Loss: 1.1352
Epoch [12/30], Loss: 1.0973
Epoch [13/30], Loss: 1.0840
Epoch [14/30], Loss: 1.1070
Epoch [15/30], Loss: 1.0825
Epoch [16/30], Loss: 1.1450
Epoch [17/30], Loss: 1.1246
Epoch [18/30], Loss: 1.1349
Epoch [19/30], Loss: 1.0612
Epoch [20/30], Loss: 1.1084
Epoch [21/30], Loss: 1.1007
Epoch [22/30], Loss: 1.1088
Epoch [23/30], Loss: 1.0751
Epoch [24/30], Loss: 1.0926
Epoch [25/30], Loss: 1.1150
Epoch [26/30], Loss: 1.0925
Epoch [27/30], Loss: 1.1215
Epoch [28/30], Loss: 1.0983
Epoch [29/30], Loss: 1.0829
Epoch [30/30], Loss: 1.0519
✅ Encoder saved as ssl_encoder.pth


In [28]:
# Save features for later evaluation
save_ssl_features(simclr, eval_loader, device=device, output_path="ssl_features.pkl")

✅ Saved SSL features to ssl_features.pkl
